# Negotiation and Bargaining: Interactive Visual Explorer

> Agents negotiate resources, prices, task allocations, and terms. The 2026 benchmark set is clear: NegotiationArena (arXiv:2402.05863) shows LLMs can improve payoffs ~20% via persona manipulation ("desperation"); "Measuring Bargaining Abilities" (arXiv:2402.15813) shows buyer is harder than seller and scale does not help — their **OG-Narrator** (deterministic offer generator + LLM narrator) pushed deal rate from 26.67% to 88.88%; the Large-Scale Autonomous Negotiation Competition (arXiv:2503.06416) ran ~180k negotiations and found that **chain-of-thought-concealing** agents win by hiding reasoning from counterparts; Bhattacharya et al. 2025 on Harvard Negotiation Project metrics ranked Llama-3 most-effective, Claude-3 aggressive, GPT-4 fairest. This lesson implements Contract Net Protocol (the FIPA ancestor, Lesson 02), wires an LLM-style buyer/seller, runs an OG-Narrator-style decomposition, and measures how deal rate changes with each structural choice.

Welcome to the interactive companion notebook for **Negotiation and Bargaining**.

In this notebook, you can interactively execute the lesson's raw implementation, plot state transformations, and run experiment variations.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11


In [ ]:
"""Negotiation: Contract Net + OG-Narrator demo, stdlib only.

Compares naive all-LLM bargaining against OG-Narrator (deterministic offer
generator + LLM narration). Measures deal rate over 1000 trials. Includes
a small Contract Net task-market demo at the end.
"""
from __future__ import annotations

import random
from dataclasses import dataclass, field

@dataclass
class BargainState:
    buyer_max: int
    seller_min: int
    buyer_offer: int | None = None
    seller_offer: int | None = None
    rounds: int = 0
    max_rounds: int = 5


In [ ]:
def naive_llm_bargain(state: BargainState, rng: random.Random) -> int:
    """Simulate naive LLM bargaining: picks a price with high variance and
    frequently outside the ZOPA (mimics strategic errors documented in
    arXiv:2402.15813)."""
    r = rng.random()
    if state.seller_offer is None:
        candidate = rng.randint(state.buyer_max - 60, state.buyer_max + 30)
    elif r < 0.35:
        candidate = state.seller_offer + rng.randint(-8, 3)
    elif r < 0.65:
        candidate = rng.randint(state.seller_min - 30, state.buyer_max + 30)
    else:
        candidate = rng.randint(state.seller_min - 60, state.buyer_max + 60)
    return candidate


In [ ]:
def og_narrator_bargain(state: BargainState, rng: random.Random,
                        concession: float = 0.35) -> int:
    """OG-Narrator deterministic offer: zeuthen-style concession toward midpoint."""
    if state.seller_offer is None and state.buyer_offer is None:
        return state.buyer_max - max(1, int((state.buyer_max - state.seller_min) * 0.2))
    if state.seller_offer is None:
        return state.buyer_offer
    prior = state.buyer_offer if state.buyer_offer is not None else state.buyer_max
    move = max(1, int(concession * (state.seller_offer - prior)))
    candidate = prior + move
    candidate = min(candidate, state.buyer_max)
    return candidate


In [ ]:
def seller_response(state: BargainState, rng: random.Random,
                    concession: float = 0.3) -> int:
    """Seller uses OG-Narrator-style offer too (for both buyers)."""
    if state.buyer_offer is None and state.seller_offer is None:
        return state.seller_min + max(1, int((state.buyer_max - state.seller_min) * 0.4))
    if state.buyer_offer is None:
        return state.seller_offer
    prior = state.seller_offer if state.seller_offer is not None else state.seller_min + 20
    move = max(1, int(concession * (prior - state.buyer_offer)))
    candidate = prior - move
    candidate = max(candidate, state.seller_min)
    return candidate


In [ ]:
def simulate_bargain(buyer_fn, rng: random.Random, buyer_max: int = 100,
                     seller_min: int = 60) -> bool:
    state = BargainState(buyer_max=buyer_max, seller_min=seller_min)
    deal = False
    while state.rounds < state.max_rounds:
        state.buyer_offer = buyer_fn(state, rng)
        if state.seller_offer is not None and state.buyer_offer >= state.seller_offer:
            # trade clears at seller's standing ask; feasible iff within both reservations
            if state.seller_offer >= state.seller_min and state.seller_offer <= state.buyer_max:
                deal = True
            break
        state.seller_offer = seller_response(state, rng)
        if state.buyer_offer is not None and state.seller_offer <= state.buyer_offer:
            # trade clears at buyer's standing bid; feasible iff within both reservations
            if state.buyer_offer <= state.buyer_max and state.buyer_offer >= state.seller_min:
                deal = True
            break
        state.rounds += 1
    return deal


In [ ]:
def bench_deal_rate(buyer_fn, label: str, trials: int = 1000) -> None:
    rng = random.Random(42)
    deals = 0
    for _ in range(trials):
        seller_min = rng.randint(50, 80)
        buyer_max = rng.randint(max(seller_min + 5, 75), 115)
        if simulate_bargain(buyer_fn, rng, buyer_max=buyer_max, seller_min=seller_min):
            deals += 1
    print(f"  {label:20s} deal rate: {deals / trials:.2%}  ({deals}/{trials})")


In [ ]:
@dataclass
class Bid:
    bidder: str
    price: int
    eta_minutes: int
    confidence: float

@dataclass
class ContractNetTask:
    task_id: str
    description: str
    deadline_minutes: int
    budget: int

class ContractNetManager:
    def __init__(self, bidders: list[str]) -> None:
        self.bidders = bidders
        self.proposals: dict[str, list[Bid]] = {}

def broadcast_cfp(self, task: ContractNetTask) -> None:
        self.proposals[task.task_id] = []
        print(f"  manager CFP -> {task.description} (deadline {task.deadline_minutes}m, budget {task.budget})")


In [ ]:
def receive_proposal(self, task_id: str, bid: Bid) -> None:
        self.proposals[task_id].append(bid)
        print(f"    propose from {bid.bidder}: price={bid.price} eta={bid.eta_minutes}m conf={bid.confidence:.2f}")

def award(self, task: ContractNetTask) -> Bid | None:
        props = self.proposals.get(task.task_id, [])
        feasible = [b for b in props if b.price <= task.budget and b.eta_minutes <= task.deadline_minutes]
        if not feasible:
            print("    no feasible bid; awarding rejected")
            return None
        winner = max(feasible, key=lambda b: b.confidence / max(b.price, 1))
        print(f"  manager accept-proposal -> {winner.bidder} (score = conf/price)")
        for b in props:
            if b is not winner:
                print(f"  manager reject-proposal -> {b.bidder}")
        return winner


In [ ]:
def demo_contract_net() -> None:
    print("\n" + "=" * 72)
    print("CONTRACT NET TASK MARKET — manager + 3 bidders")
    print("=" * 72)
    task = ContractNetTask(
        task_id="t-1",
        description="compress 10GB log bundle",
        deadline_minutes=30,
        budget=10,
    )
    mgr = ContractNetManager(bidders=["worker-a", "worker-b", "worker-c"])
    mgr.broadcast_cfp(task)
    mgr.receive_proposal(task.task_id, Bid("worker-a", price=3, eta_minutes=18, confidence=0.82))
    mgr.receive_proposal(task.task_id, Bid("worker-b", price=2, eta_minutes=25, confidence=0.77))
    mgr.receive_proposal(task.task_id, Bid("worker-c", price=4, eta_minutes=10, confidence=0.90))
    mgr.award(task)


In [ ]:
def main() -> None:
    print("=" * 72)
    print("DEAL RATE — naive LLM bargaining vs OG-Narrator")
    print("reservation prices sampled per trial: seller_min in [50,80], buyer_max in [75,115]")
    print("=" * 72)
    bench_deal_rate(naive_llm_bargain, "naive LLM")
    bench_deal_rate(og_narrator_bargain, "OG-Narrator")
    demo_contract_net()

print("\nTakeaways:")
    print("  naive LLM-only bargaining has inflated variance -- swings outside the ZOPA.")
    print("  OG-Narrator (deterministic offer + LLM narration) converges on every trial")
    print("  because prices are arithmetic, not generative.")
    print("  The original paper (arXiv:2402.15813) reports 26.67% -> 88.88% on the tighter")
    print("  real-LLM benchmark. Our simulation shrinks the gap because the opposing seller")
    print("  is already using deterministic offers -- the structural pattern is the same.")
    print("  Contract Net scales: broadcast + collect + award; no synchronous chat needed.")


In [ ]:
if __name__ == "__main__":
    main()
